# Quadcopter Structures Lab: General Loading and 3D Deformation

**Google Colab | Linear isotropic elasticity | Cantilever beam**

A quadcopter arm transmits forces and moments from the motor assembly to the airframe. Vertical rotor thrust primarily causes bending, while motor reaction torque acts about the rotor axis and may cause bending in a different plane. Other loading conditions can introduce axial deformation or torsion.

In this lab, we analyze a cantilever beam subjected to a general force and moment at its free end:

$$
\mathbf p =
\begin{bmatrix}F_x&F_y&F_z&M_x&M_y&M_z\end{bmatrix}^{T}.
$$

**Coordinate convention (consistent with the previous labs):** $x$ is along the arm, $y$ is vertical, and $z$ is lateral. Input forces are in grams-force (gf) and input moments in gf mm, consistent with our earlier labs. The solver converts these to newtons (N) and newton-meters (N m) internally. Positive moments follow the right-hand rule.

**Companion instructions:** The step-by-step assignment, requested results, and submission requirements are in `README.md`. This notebook provides the analytical reference equations, numerical model, and figures used for those tasks.

**Running the notebook:** In Google Colab, choose **Runtime → Run all** for the default settings. Change the geometry, material, or `LOAD_CASE` in Section 2 for a new calculation and rerun from that cell. Input units are SI (m, N, Pa).

**Physical interpretation:** For a motor with a vertical shaft, thrust is represented by $F_y$ and the motor reaction couple by $M_y$. The latter causes lateral bending, not torsion of the arm. Torsion about the longitudinal axis corresponds to $M_x$.


## 1. Software and finite-element discretization

This notebook uses **NumPy** for numerical calculations, **Matplotlib** for visualization, and **scikit-fem** to assemble and solve the three-dimensional elasticity problem.

The arm is discretized into tetrahedral finite elements. We use quadratic displacement interpolation (`ElementTetP2`), which is generally better suited to resolving bending than first-order tetrahedral elements at comparable mesh densities.

Run the installation and import cells below; no modifications are required.


In [ ]:
%pip -q install "scikit-fem==12.0.2"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.sparse.linalg import splu
import skfem
from skfem import (MeshTet, Basis, FacetBasis, ElementVector, ElementTetP2,
                   BilinearForm, LinearForm, Functional, asm, condense, solve)
from skfem.helpers import sym_grad, ddot, trace, eye

np.set_printoptions(precision=5, suppress=True)
print('scikit-fem version:', skfem.__version__)

## 2. Geometry, material properties, and loading

The cantilever is clamped at $x=0$ and loaded at $x=L$. We use the same **120 mm arm length** and **250 gf nominal thrust** as our earlier structural-analysis exercises. For this lab, everyone uses the same prescribed solid rectangular section, rather than the individually chosen plate dimensions from the earlier design work:

- $L$: arm length along $x$;
- $b$: width along the lateral $z$-axis;
- $h$: thickness along the vertical $y$-axis;
- $E$: Young's modulus; $\nu$: Poisson's ratio.

The section properties are

$$
A=bh,\qquad I_y=\frac{hb^3}{12},\qquad I_z=\frac{bh^3}{12}.
$$

The bending stiffnesses are $EI_y$ and $EI_z$. Vertical thrust $F_y$ produces bending about $z$, so the corresponding stiffness is $EI_z$.

**Inputs and units:** Enter $L$, $b$, and $h$ in **mm**, $E$ in **GPa**, forces in **gf**, and moments in **gf mm**. Conversion to SI units is handled by the notebook. Displacements are reported in mm, rotations in degrees, and stresses in gf/mm².

### Prescribed loading conditions

Set `LOAD_CASE` to select the nonzero tip loads. The default force magnitude is **250 gf**, while the motor reaction moment is calculated from the prescribed moment-to-thrust ratio of **15 mm**.

| `LOAD_CASE` | Applied tip load | Dominant response |
|---|---|---|
| `"Fx"` | $F_x=250$ gf | Axial extension |
| `"Fy"` | $F_y=250$ gf | Vertical bending |
| `"Fz"` | $F_z=250$ gf | Lateral bending |
| `"Mx"` | $M_x=2000$ gf mm | Torsion about $x$ |
| `"My"` | $M_y=3750$ gf mm | Lateral bending due to an end moment |
| `"Mz"` | $M_z=3750$ gf mm | Vertical bending due to an end moment |
| `"Fy+My"` | $F_y=250$ gf and $M_y=3750$ gf mm | Bending in two planes |
| `"all"` | All six components | General combined loading |
| `"custom"` | Values entered in `CUSTOM_LOAD_GFM` | User-defined loading |

In the combined motor-load case, $M_y$ is a **signed reaction moment** about the vertical rotor axis. Its direction depends on propeller rotation. The positive sign is prescribed here for consistency.


In [ ]:
# ============== STUDENT INPUTS (same units as earlier labs) ==============
# Geometry [mm]: L is the SAME 120 mm length used previously.
L_mm = 120.0                # arm length along x [mm]
b_mm = 20.0                 # width along z [mm]
h_mm = 10.0                 # thickness along y [mm]

# Material: illustrative isotropic polymer (not a specific manufactured plate)
E_GPa = 3.0                 # Young's modulus [GPa]
nu = 0.35                   # Poisson's ratio [-]

# Loading inputs: grams-force (gf) and gf mm.
# 250 gf is the nominal motor thrust from the earlier truss analysis.
T_gf = 250.0                # force magnitude [gf]
torque_ratio_mm = 15.0      # rotor reaction moment / thrust [mm]
Mx_gf_mm = 2000.0           # pure torsional moment for separate exercise [gf mm]

# Number of mesh divisions along x, y, z.
nx, ny, nz = 16, 4, 4

# Choose: 'Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz', 'Fy+My', 'all', 'custom'
LOAD_CASE = 'Fy'

# Only used with LOAD_CASE = 'custom'. Components are signed.
#                   Fx    Fy    Fz     Mx     My     Mz
# Units:             gf    gf    gf    gf mm  gf mm  gf mm
CUSTOM_LOAD_GFM = [0.0, 250.0, 0.0, 0.0, 0.0, 0.0]

# ============== PROVIDED MODEL CALCULATIONS ================
GF_TO_N = 0.00980665          # exactly 1 gf = 0.00980665 N
GF_MM_TO_N_M = GF_TO_N * 1e-3
PA_TO_GF_MM2 = 1e-6 / GF_TO_N
L, b, h = L_mm*1e-3, b_mm*1e-3, h_mm*1e-3
E = E_GPa*1e9
T_N = T_gf*GF_TO_N
Q_gf_mm = torque_ratio_mm*T_gf
Q_Nm = Q_gf_mm*GF_MM_TO_N_M
Mx_Nm = Mx_gf_mm*GF_MM_TO_N_M

def gfmm_to_si(loads):
    """Convert [Fx,Fy,Fz,Mx,My,Mz] from gf/gf mm to N/N m."""
    v = np.asarray(loads, dtype=float)
    return v * np.array([GF_TO_N]*3 + [GF_MM_TO_N_M]*3)

LOAD_CASES = {
    'Fx':    gfmm_to_si([T_gf, 0, 0, 0, 0, 0]),
    'Fy':    gfmm_to_si([0, T_gf, 0, 0, 0, 0]),
    'Fz':    gfmm_to_si([0, 0, T_gf, 0, 0, 0]),
    'Mx':    gfmm_to_si([0, 0, 0, Mx_gf_mm, 0, 0]),
    'My':    gfmm_to_si([0, 0, 0, 0, Q_gf_mm, 0]),
    'Mz':    gfmm_to_si([0, 0, 0, 0, 0, Q_gf_mm]),
    'Fy+My':gfmm_to_si([0, T_gf, 0, 0, Q_gf_mm, 0]),
    'all':   gfmm_to_si([T_gf/2, T_gf, -T_gf/2,
                        Mx_gf_mm, Q_gf_mm, -Q_gf_mm]),
    'custom':gfmm_to_si(CUSTOM_LOAD_GFM),
}
assert LOAD_CASE in LOAD_CASES, 'Select a load name from the table above.'
assert min(L, b, h, E) > 0 and -1.0 < nu < 0.49
assert min(nx, ny, nz) >= 2
assert T_gf >= 0 and torque_ratio_mm >= 0

p = np.asarray(LOAD_CASES[LOAD_CASE], dtype=float)
F_tip, M_tip = p[:3], p[3:]
G = E/(2.0*(1.0+nu))
A, Iy, Iz = b*h, h*b**3/12.0, b*h**3/12.0
Ip = Iy + Iz

# Approximate Saint-Venant torsion constant for a solid rectangle.
a, c = max(b, h), min(b, h)
J = a*c**3 * (1.0/3.0 - 0.21*(c/a)*(1.0-(c/a)**4/12.0))
root_moment = M_tip + np.cross(np.array([L, 0., 0.]), F_tip)

print(f'Geometry: L={L_mm:g} mm, b={b_mm:g} mm, h={h_mm:g} mm')
print(f'Material: E={E_GPa:g} GPa, nu={nu:g}')
print(f'Thrust: {T_gf:g} gf = {T_N:.6f} N')
print(f'Reaction moment: (Q/T)={torque_ratio_mm:g} mm -> '
      f'Q={Q_gf_mm:g} gf mm = {Q_Nm:.6f} N m')
print('Selected load:', LOAD_CASE)
print('Tip force [Fx,Fy,Fz] (N):', F_tip)
print('Tip moment [Mx,My,Mz] (N m):', M_tip)
print('Moment of tip loads about clamp (N m):', root_moment)
print(f'A = {A:.3e} m^2; Iy = {Iy:.3e} m^4; Iz = {Iz:.3e} m^4')
print(f'J = {J:.3e} m^4; G = {G/1e9:.3f} GPa')


## 3. Analytical prediction: the beam compliance matrix

We first calculate the response using **Euler–Bernoulli beam theory** for axial deformation and bending, together with **Saint-Venant torsion**. The relevant equations are supplied; the emphasis is on applying them and comparing their predictions with FEA.

At a section located at $x$, the tip loads produce the moment resultants

$$
\begin{aligned}
m_x(x)&=M_x,\\
m_y(x)&=M_y-(L-x)F_z,\\
m_z(x)&=M_z+(L-x)F_y.
\end{aligned}
$$

The free-end response comprises three translations and three rotations:

$$
\mathbf q =
\begin{bmatrix}u_L&v_L&w_L&\theta_{x,L}&\theta_{y,L}&\theta_{z,L}\end{bmatrix}^{T}.
$$

The relationship between the applied loads and the tip response is

$$\boxed{\mathbf q=\mathbf H\mathbf p},$$

where $\mathbf H$ is the **compliance matrix**. Each entry gives the response in one degree of freedom to a unit generalized load; the corresponding stiffness matrix is $\mathbf H^{-1}$.

For a uniform isotropic beam with centroidal principal bending axes,

$$
\mathbf H =
\begin{bmatrix}
\dfrac{L}{EA} & 0 & 0 & 0 & 0 & 0 \\
0 & \dfrac{L^3}{3EI_z} & 0 & 0 & 0 & \dfrac{L^2}{2EI_z} \\
0 & 0 & \dfrac{L^3}{3EI_y} & 0 & -\dfrac{L^2}{2EI_y} & 0 \\
0 & 0 & 0 & \dfrac{L}{GJ} & 0 & 0 \\
0 & 0 & -\dfrac{L^2}{2EI_y} & 0 & \dfrac{L}{EI_y} & 0 \\
0 & \dfrac{L^2}{2EI_z} & 0 & 0 & 0 & \dfrac{L}{EI_z}
\end{bmatrix}.
$$

The rows correspond to $u_L,v_L,w_L,\theta_{x,L},\theta_{y,L},\theta_{z,L}$; the columns correspond to $F_x,F_y,F_z,M_x,M_y,M_z$.

For example, the $F_y$ column predicts vertical deflection $v_L$ and rotation $\theta_{z,L}$, but no longitudinal twist $\theta_{x,L}$. The off-diagonal terms capture the coupling between a transverse displacement and the associated bending rotation.

The torsion constant $J$ is not generally the polar second moment $I_p=I_y+I_z$ for a noncircular cross-section. We calculate $J$ using an approximation for a solid rectangular section.

**Model assumptions:** linear elasticity; small strains and rotations; a straight, prismatic beam; isotropic material properties; and negligible transverse shear deformation and additional effects of constrained warping. We will evaluate the accuracy of these approximations using FEA.


In [ ]:
def beam_compliance(L, E, G, A, Iy, Iz, J):
    H = np.zeros((6, 6))
    H[0, 0] = L/(E*A)
    H[1, 1] = L**3/(3*E*Iz)
    H[2, 2] = L**3/(3*E*Iy)
    H[3, 3] = L/(G*J)
    H[4, 4] = L/(E*Iy)
    H[5, 5] = L/(E*Iz)
    H[1, 5] = H[5, 1] = L**2/(2*E*Iz)
    H[2, 4] = H[4, 2] = -L**2/(2*E*Iy)
    return H

H_beam = beam_compliance(L, E, G, A, Iy, Iz, J)
q_beam = H_beam @ p
q_labels = ['u_L [mm]', 'v_L [mm]', 'w_L [mm]',
            'theta_x [deg]', 'theta_y [deg]', 'theta_z [deg]']
q_display_scale = np.array([1e3, 1e3, 1e3, 180/np.pi, 180/np.pi, 180/np.pi])
print('PREDICTED TIP MOVEMENT (BEAM EQUATIONS)')
for label, val in zip(q_labels, q_beam*q_display_scale):
    print(f'{label:17s} {val: .6f}')

# Display the numerical compliance matrix in scientific notation.
# Rows: u, v, w, theta_x, theta_y, theta_z; columns: Fx, Fy, Fz, Mx, My, Mz.
with np.printoptions(precision=3, suppress=False):
    print('Numerical H matrix (SI units):')
    print(H_beam)


## 4. Three-dimensional finite-element mesh

The finite-element mesh divides the beam into tetrahedra, with displacement approximated within each element. Increasing `nx`, `ny`, or `nz` refines the discretization, usually improving accuracy at increased computational cost.

We impose a **clamped boundary condition** on the root face ($x=0$): all three displacement components vanish there. The prescribed tip forces and moments are applied on the opposite face ($x=L$).

The following cells construct and display the mesh.


In [ ]:
# Structured tetrahedral mesh of the solid arm
x_nodes = np.linspace(0., L, nx+1)
y_nodes = np.linspace(-h/2, h/2, ny+1)
z_nodes = np.linspace(-b/2, b/2, nz+1)
mesh = MeshTet.init_tensor(x_nodes, y_nodes, z_nodes)

# Use P2 (quadratic) tetrahedral displacement elements
scalar_element = ElementTetP2()
element = ElementVector(scalar_element)
basis = Basis(mesh, element)
right_facets = mesh.facets_satisfying(
    lambda X: np.isclose(X[0], L), boundaries_only=True)
end_basis = FacetBasis(mesh, element, facets=right_facets)
fixed_dofs = basis.get_dofs(lambda X: np.isclose(X[0], 0.))

surface_triangles = mesh.facets[:, mesh.boundary_facets()].T
print(f'Mesh: {mesh.t.shape[1]} tetrahedra, {mesh.p.shape[1]} geometric vertices')
print(f'Displacement DOFs: {basis.N}; clamped DOFs: {len(fixed_dofs.all())}')

In [ ]:
# 3D mesh overview (outer surface; all lengths in mm for this plot)
fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(111, projection='3d')
faces = (mesh.p[:, surface_triangles].transpose(1, 2, 0) * 1e3)
poly = Poly3DCollection(faces, facecolors='lightskyblue', edgecolors='dimgray',
                        linewidths=0.16, alpha=0.32)
ax.add_collection3d(poly)
ax.set(xlim=(0, L*1e3), ylim=(-h*0.8e3, h*0.8e3),
       zlim=(-b*0.9e3, b*0.9e3),
       xlabel=r'$x$ [mm]', ylabel=r'$y$ [mm]', zlabel=r'$z$ [mm]')
ax.set_box_aspect((5, 1.4, 1.0))
ax.view_init(elev=22, azim=-65)
ax.set_title('3D tetrahedral mesh: fixed at $x=0$, loaded at $x=L$')
plt.tight_layout(); plt.show()

## 5. Applying forces and moments to a solid-element model

The finite-element unknowns are **nodal translations**; the solid elements do not have independent rotational degrees of freedom. We therefore represent an applied tip couple by a distribution of surface forces on the end face.

Let $\mathbf t=(t_x,t_y,t_z)$ denote the **traction** (force per unit area). For the rectangular cross-section, we use

$$
\begin{aligned}
t_x(y,z)&=\frac{F_x}{A}+\frac{M_y z}{I_y}-\frac{M_z y}{I_z},\\
t_y(y,z)&=\frac{F_y}{A}-\frac{M_x z}{I_p},\\
t_z(y,z)&=\frac{F_z}{A}+\frac{M_x y}{I_p},
\end{aligned}
$$

where $I_p=I_y+I_z$. The resulting force and moment about the tip-face centroid satisfy

$$
\int_A\mathbf t\,dA=\mathbf F,
\qquad
\int_A\mathbf r\times\mathbf t\,dA=\mathbf M,
\qquad \mathbf r=(0,y,z).
$$

The notebook evaluates these resultants numerically as a check on the applied loading. Note that **$I_p$ and $J$ serve different purposes**: $I_p$ appears in this chosen traction distribution, whereas the torsional stiffness in beam theory is $GJ$. The traction field has the correct resultant moment but is not necessarily the exact Saint-Venant torsional traction distribution near the tip.


In [ ]:
def tip_traction(y, z, loads):
    """Traction components (Pa) producing prescribed (Fx,Fy,Fz,Mx,My,Mz)."""
    Fx, Fy, Fz, Mx, My, Mz = loads
    tx = Fx/A + My*z/Iy - Mz*y/Iz
    ty = Fy/A - Mx*z/Ip
    tz = Fz/A + Mx*y/Ip
    return tx, ty, tz

def make_end_form(loads):
    @LinearForm
    def end_load(v, w):
        tx, ty, tz = tip_traction(w.x[1], w.x[2], loads)
        return tx*v[0] + ty*v[1] + tz*v[2]
    return end_load

unit_loads = np.eye(6)
B_load = np.column_stack([
    asm(make_end_form(unit_loads[:, j]), end_basis) for j in range(6)
])
f = B_load @ p

def integrated_resultants(loads):
    # Numerically integrate the traction resultants over the end face.
    def component(j):
        @Functional
        def value(w):
            y, z = w.x[1], w.x[2]
            tx, ty, tz = tip_traction(y, z, loads)
            return (tx, ty, tz, y*tz-z*ty, z*tx, -y*tx)[j]
        return value.assemble(end_basis)
    return np.array([component(j) for j in range(6)])

p_check = integrated_resultants(p)
print('Requested [Fx Fy Fz Mx My Mz]:', p)
print('Integrated end-face resultants:', p_check)
np.testing.assert_allclose(p_check, p, rtol=1e-9, atol=1e-9)
print('End load balance: PASS')

## 6. Solving the three-dimensional elasticity problem

For a linearly elastic solid, finite-element discretization gives the linear system

$$\mathbf K\mathbf d=\mathbf f,$$

where $\mathbf K$ is the assembled stiffness matrix, $\mathbf d$ contains the unknown nodal displacements, and $\mathbf f$ is the applied nodal-force vector (including the equivalent end-face loads).

The supplied code assembles $\mathbf K$, enforces the clamped boundary conditions, and solves for $\mathbf d$. It also checks the elastic energy balance $\tfrac12\mathbf d^T\mathbf K\mathbf d=\tfrac12\mathbf d^T\mathbf f$ for the computed equilibrium solution.


In [ ]:
mu = G
lam = E*nu / ((1.+nu)*(1.-2.*nu))

@BilinearForm
def elasticity(u, v, w):
    eu, ev = sym_grad(u), sym_grad(v)
    su = 2.*mu*eu + lam*eye(trace(eu), 3)
    return ddot(su, ev)

K = asm(elasticity, basis)

displacement = solve(*condense(K, f, D=fixed_dofs))
u_nodes = displacement[basis.nodal_dofs]  # (3, nvertices): only geometric vertices for plots
ux, uy, uz = u_nodes
print('FEA solve complete.')
print(f'Maximum vertex displacement magnitude = '
      f'{np.linalg.norm(u_nodes, axis=0).max()*1e3:.6f} mm')

# Check global elastic energy equals work of the applied forces (linear problem).
energy_strain = 0.5 * displacement @ (K @ displacement)
energy_load = 0.5 * displacement @ f
print(f'Strain energy = {energy_strain:.6e} J;  1/2 f^T d = {energy_load:.6e} J')
np.testing.assert_allclose(energy_strain, energy_load, rtol=1e-7, atol=1e-11)

## 7. Comparison of analytical and finite-element tip responses

The next table reports the six tip responses predicted by beam theory and extracted from the three-dimensional displacement field.

Two FEA measures are included:

- **FEA weighted:** A generalized displacement or rotation obtained from the end-face displacement field using the same loading distributions as in Section 5. This measure is consistent with the applied loads and is useful for constructing the numerical compliance matrix.
- **FEA face fit:** Translation and rotation estimated by fitting a small rigid-body motion to the end-face nodal displacements. This gives a geometric description of the tip-face motion.

The two measures need not be identical because a three-dimensional solid section can deform and warp rather than move as a perfectly rigid plane.

The relative agreement of these estimates with beam theory depends on the loading, beam slenderness, and the details of the three-dimensional end-face deformation.


In [ ]:
# Work-conjugate generalized tip motion
q_FE = B_load.T @ displacement

# Fit the end-face motion to u0 + theta cross r, using the mesh vertices.
end_vertices = np.where(np.isclose(mesh.p[0], L))[0]
r_tip = mesh.p[:, end_vertices].T.copy()
r_tip[:, 0] = 0.0             # offsets from centroid of x=L section
U_tip = u_nodes[:, end_vertices].T
D = np.zeros((3*len(end_vertices), 6))
for i, (_, y, z) in enumerate(r_tip):
    D[3*i:3*i+3, :] = [[1, 0, 0,  0,  z, -y],
                       [0, 1, 0, -z,  0,  0],
                       [0, 0, 1,  y,  0,  0]]
q_fit = np.linalg.lstsq(D, U_tip.reshape(-1), rcond=None)[0]

print('Six free-end responses (translations in mm; rotations in deg):')
print(f'{"Response":18s} {"Beam theory":>14s} {"FEA weighted":>14s} {"FEA face fit":>14s}')
for i, label in enumerate(q_labels):
    s = q_display_scale[i]
    print(f'{label:18s} {q_beam[i]*s:14.6f} '
          f'{q_FE[i]*s:14.6f} {q_fit[i]*s:14.6f}')

## 8. Displacement distribution along the arm

We now compare the **centerline displacement** along the beam rather than only at the free end. The analytical expressions are

$$
\begin{aligned}
u(x)&=\frac{F_xx}{EA},\\
v(x)&=\frac{F_yx^2(3L-x)}{6EI_z}+\frac{M_zx^2}{2EI_z},\\
w(x)&=\frac{F_zx^2(3L-x)}{6EI_y}-\frac{M_yx^2}{2EI_y}.
\end{aligned}
$$

The curves represent beam theory; the plotted FEA points are cross-section averages of the nodal displacements at each station $x$. This averaging approximates the displacement of the section centroid.

For pure torsion (`"Mx"`), the centerline translations are approximately zero, although the cross-sections rotate about $x$. Section 11 examines this rotation separately.


In [ ]:
x_stations = np.unique(np.round(mesh.p[0], 12))
means = np.column_stack([
    u_nodes[:, np.isclose(mesh.p[0], xi)].mean(axis=1)
    for xi in x_stations
]).T
u_bar = F_tip[0]*x_stations/(E*A)
v_bar = (F_tip[1]*x_stations**2*(3*L-x_stations)/(6*E*Iz)
         + M_tip[2]*x_stations**2/(2*E*Iz))
w_bar = (F_tip[2]*x_stations**2*(3*L-x_stations)/(6*E*Iy)
         - M_tip[1]*x_stations**2/(2*E*Iy))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True)
for j, (ax, theory, name) in enumerate(zip(axes, [u_bar, v_bar, w_bar],
                                           [r'$u_x$',r'$u_y$',r'$u_z$'])):
    ax.plot(x_stations*1e3, theory*1e3, 'k-', lw=2, label='Beam theory')
    ax.plot(x_stations*1e3, means[:,j]*1e3, 'o', ms=3, label='3D FEA')
    ax.set(xlabel=r'$x$ [mm]', ylabel='Displacement [mm]', title=name)
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=8)
fig.suptitle('Cross-sectional mean displacement versus position')
plt.tight_layout(); plt.show()

## 9. Three-dimensional deformation

The figure superposes the undeformed and deformed beam surfaces. Displacements are **magnified for visualization**, and the scale factor is displayed in the plot title; the shape does not represent the actual deformation at full scale.

Compare the displacement patterns for `"Fy"` (vertical bending), `"My"` (lateral bending from an end couple), and `"Mx"` (torsion). In particular, examine the centerline displacement and the orientation of the free-end cross-section.


In [ ]:
max_u = np.linalg.norm(u_nodes, axis=0).max()
scale = (0.12*L/max_u) if max_u > 0 else 1.0
# Torsional rotation can be appreciable even for small nodal displacements.
# Prevent the magnification from giving visually inverted/folded cross sections.
max_rot = np.max(np.abs(q_fit[3:]))
if max_rot > 0:
    scale = min(scale, 0.30/max_rot)
p_deformed = mesh.p + scale*u_nodes
faces_ref = mesh.p[:, surface_triangles].transpose(1, 2, 0)*1e3
faces_def = p_deformed[:, surface_triangles].transpose(1, 2, 0)*1e3

fig = plt.figure(figsize=(11, 5))
ax = fig.add_subplot(111, projection='3d')
ax.add_collection3d(Poly3DCollection(faces_ref, facecolors='gray', edgecolors='gray',
                                    linewidths=0.10, alpha=0.12))
ax.add_collection3d(Poly3DCollection(faces_def, facecolors='cornflowerblue',
                                    edgecolors='navy', linewidths=0.13, alpha=0.55))
combined_p = np.hstack([mesh.p, p_deformed])*1e3
for i, setter in enumerate([ax.set_xlim, ax.set_ylim, ax.set_zlim]):
    low, high = np.min(combined_p[i]), np.max(combined_p[i])
    margin = max(0.05*(high-low), 0.8)
    setter(low-margin, high+margin)
ax.set(xlabel=r'$x$ [mm]', ylabel=r'$y$ [mm]', zlabel=r'$z$ [mm]')
ax.set_box_aspect((5, 1.8, 1.8))
ax.view_init(elev=23, azim=-65)
ax.set_title(rf'Undeformed (gray) and deformed (blue), amplification $\times {scale:.1f}$')
plt.tight_layout(); plt.show()

## 10. Longitudinal normal stress

The longitudinal normal stress $\sigma_{xx}$ combines axial loading with bending about the $y$ and $z$ axes. Positive $\sigma_{xx}$ indicates tension and negative $\sigma_{xx}$ indicates compression.

The analytical beam model gives

$$
\sigma_{xx}(x,y,z)=\frac{F_x}{A}
+\frac{[M_y-(L-x)F_z]z}{I_y}
-\frac{[M_z+(L-x)F_y]y}{I_z}.
$$

The following plots compare this expression with the finite-element stress at the beam midpoint ($x=L/2=60$ mm for the nominal geometry), as in our earlier stress analysis. FEA stresses are evaluated from the computed strain field and shown in gf/mm².

Differences are expected near the clamped boundary and loaded face, where local three-dimensional stress distributions are not fully represented by elementary beam theory.


In [ ]:
# Recover full 3D stress components from the FE displacement gradients.
uh = basis.interpolate(displacement)
Grad = uh.grad
exx, eyy, ezz = Grad[0,0], Grad[1,1], Grad[2,2]
exy = 0.5*(Grad[0,1]+Grad[1,0])
exz = 0.5*(Grad[0,2]+Grad[2,0])
eyz = 0.5*(Grad[1,2]+Grad[2,1])
tr_eps = exx+eyy+ezz
sigma_xx = 2*mu*exx + lam*tr_eps
sigma_yy = 2*mu*eyy + lam*tr_eps
sigma_zz = 2*mu*ezz + lam*tr_eps
tau_xy = 2*mu*exy
tau_xz = 2*mu*exz
tau_yz = 2*mu*eyz
# For visualization only: average each component over element quadrature points.
mean_el = lambda a: np.mean(a, axis=1)
sxx_e, syy_e, szz_e = map(mean_el, (sigma_xx, sigma_yy, sigma_zz))
txy_e, txz_e, tyz_e = map(mean_el, (tau_xy, tau_xz, tau_yz))
vm_e = np.sqrt(0.5*((sxx_e-syy_e)**2 + (syy_e-szz_e)**2 + (szz_e-sxx_e)**2)
               + 3*(txy_e**2+txz_e**2+tyz_e**2))

# Tetrahedron centroids identify a narrow slab around x_section.
centers = mesh.p[:, mesh.t].mean(axis=1)
x_section = 0.50*L
slab = np.abs(centers[0]-x_section) < 0.60*(L/nx)
y_sl, z_sl = centers[1,slab], centers[2,slab]
x_sl = centers[0,slab]
my_sl = M_tip[1] - (L-x_sl)*F_tip[2]
mz_sl = M_tip[2] + (L-x_sl)*F_tip[1]
sxx_beam_sl = F_tip[0]/A + my_sl*z_sl/Iy - mz_sl*y_sl/Iz
assert len(y_sl)>0

vmin = min(sxx_e[slab].min(), sxx_beam_sl.min()) * PA_TO_GF_MM2
vmax = max(sxx_e[slab].max(), sxx_beam_sl.max()) * PA_TO_GF_MM2
if abs(vmax-vmin) < 1e-12:
    vmin, vmax = vmin-0.1, vmax+0.1
fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, values, heading in zip(axs, [sxx_beam_sl, sxx_e[slab]],
                               ['Beam theory', '3D FEA, element-averaged']):
    sc = ax.scatter(z_sl*1e3, y_sl*1e3, c=values*PA_TO_GF_MM2,
                    vmin=vmin, vmax=vmax, s=34, cmap='coolwarm')
    ax.set(xlabel=r'$z$ [mm] (lateral)', ylabel=r'$y$ [mm] (vertical)', title=heading)
    ax.set_aspect('equal', adjustable='box'); ax.grid(alpha=0.15)
fig.colorbar(sc, ax=axs, shrink=0.8, label=r'$\sigma_{xx}$ [gf/mm$^2$]')
fig.suptitle(f'Normal stress near $x={x_section*1e3:.0f}$ mm')
plt.show()
print(f'Interior-section stress samples: {len(y_sl)} elements')
print(f'Max element-average von Mises stress: {vm_e.max()*PA_TO_GF_MM2:.3f} gf/mm^2')
print('Note: peak mesh stress at an ideal clamp is not a converged failure measure.')

## 11. Torsional rotation and shear stresses

A tip moment $M_x$ produces torsion about the beam's longitudinal axis. Under Saint-Venant torsion, the predicted rotation is

$$\theta_x(x)=\frac{M_xx}{GJ}.$$

The first plot compares this prediction with the twist extracted from the FEA displacement field at successive cross-sections.

The remaining plots show the shear-stress components $\tau_{xy}$ and $\tau_{xz}$ and the **von Mises equivalent stress**, a scalar measure commonly used to assess yielding in ductile isotropic materials. The three-dimensional stress field includes effects not present in the simplified bending and torsion formulas.

For the `"Mx"` loading case, the centerline displacements in Section 8 may be small while the cross-sectional twist is substantial.


In [ ]:
# Extract section twist from the antisymmetric transverse displacement pattern.
twist_FE = np.zeros_like(x_stations)
for i, xi in enumerate(x_stations):
    nodes = np.isclose(mesh.p[0], xi)
    yc = mesh.p[1, nodes] - mesh.p[1, nodes].mean()
    zc = mesh.p[2, nodes] - mesh.p[2, nodes].mean()
    vy = uy[nodes] - uy[nodes].mean()
    vz = uz[nodes] - uz[nodes].mean()
    denominator = np.sum(yc**2 + zc**2)
    twist_FE[i] = np.sum(yc*vz - zc*vy) / denominator

twist_beam = M_tip[0]*x_stations/(G*J)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(x_stations*1e3, twist_beam*180/np.pi, 'k-', lw=2, label='Beam theory')
ax.plot(x_stations*1e3, twist_FE*180/np.pi, 'o', ms=3, label='3D FEA section fit')
ax.set(xlabel=r'$x$ [mm]', ylabel=r'$\theta_x$ [deg]',
       title='Twist of the arm about its longitudinal axis')
ax.legend(); ax.grid(alpha=0.25); plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True, sharey=True)
for k, (ax, field, heading) in enumerate(zip(axs,
    [txy_e[slab], txz_e[slab], vm_e[slab]],
    [r'$\tau_{xy}$', r'$\tau_{xz}$', r'$\sigma_{\mathrm{VM}}$'])):
    if k == 2:
        lo, hi, cmap = 0., max(0.01, field.max()*PA_TO_GF_MM2), 'viridis'
    else:
        cmax = max(0.01, np.max(np.abs(field))*PA_TO_GF_MM2)
        lo, hi, cmap = -cmax, cmax, 'coolwarm'
    im = ax.scatter(z_sl*1e3, y_sl*1e3, c=field*PA_TO_GF_MM2,
                    s=30, cmap=cmap, vmin=lo, vmax=hi)
    fig.colorbar(im, ax=ax, shrink=0.75, label=r'gf/mm$^2$')
    ax.set(xlabel=r'$z$ [mm] (lateral)', ylabel=r'$y$ [mm] (vertical)', title=heading)
    ax.set_aspect('equal', adjustable='box')
axs[0].set_ylabel(r'$y$ [mm] (vertical)')
fig.suptitle('Section stress components (element-averaged)')
plt.show()

## 12. Optional: calculating the numerical compliance matrix

Because the finite-element problem is linear, its six-by-six tip compliance matrix can be assembled from **six independent load cases**. Applying a unit load in each component of $\mathbf p$ gives one column of $\mathbf H_{\mathrm{FEA}}$:

$$\mathbf q=\mathbf H\mathbf p.$$

Set `RUN_COMPLIANCE_STUDY=True` to compute $\mathbf H_{\mathrm{FEA}}$ using the weighted generalized responses from Section 7. This requires additional solves, but the stiffness matrix is reused.

In linear elasticity the resulting FEA compliance matrix is symmetric (reciprocity). The entries have different units, so comparisons should be made between corresponding entries rather than from raw matrix magnitudes.


In [ ]:
RUN_COMPLIANCE_STUDY = False    # Set True to run the optional six-load calculation

if RUN_COMPLIANCE_STUDY:
    constrained = fixed_dofs.all()
    free = np.setdiff1d(np.arange(basis.N), constrained)
    solver = splu(K[free, :][:, free].tocsc())
    U_basis = np.zeros((basis.N, 6))
    U_basis[free, :] = solver.solve(B_load[free, :])
    H_FE = B_load.T @ U_basis
    reciprocal_error = np.linalg.norm(H_FE-H_FE.T, ord='fro') / np.linalg.norm(H_FE, ord='fro')
    print('Relative compliance reciprocity error:', reciprocal_error)
    print('Beam compliance H_beam:')
    print(H_beam)
    print('FE compliance H_FE:')
    print(H_FE)
    assert reciprocal_error < 1e-8
else:
    print('Optional matrix comparison skipped. Set RUN_COMPLIANCE_STUDY=True to run.')